# Treinamento Definitivo V3.4 (Correção do Erro)

**O que mudou agora:**
Adicionei `mode='max'` no EarlyStopping. O Keras reclamou que não sabia se devia parar quando o numero sobe ou desce. Agora eu disse explicitamente: "Pare se a acurácia parar de SUBIR".

**Boa notícia do erro:**
Vi no seu log: `val_gender_output_accuracy: 0.75116`. 
Isso é INCRÍVEL! Na primeira época ele já bateu **75% de acerto**. O modelo antigo estava travado em 50%. Significa que a estratégia funcionou, só faltou essa linha de código.

**Instruções:** Runtime > T4 GPU > Run All.

In [ ]:
import os
import numpy as np
import requests
import tarfile
import shutil
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

print("TF Version:", tf.__version__)

In [ ]:
# 2. Baixar Dataset
DATASET_URL = "https://huggingface.co/datasets/py97/UTKFace-Cropped/resolve/main/UTKFace.tar.gz"
DEST_DIR = "./data"
TAR_PATH = "./UTKFace.tar.gz"

if not os.path.exists(DEST_DIR):
    os.makedirs(DEST_DIR)

if not os.path.exists(os.path.join(DEST_DIR, "modified")):
    print("Baixando...")
    r = requests.get(DATASET_URL, stream=True)
    with open(TAR_PATH, 'wb') as f:
        f.write(r.content)
    
    print("Extraindo...")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(path=DEST_DIR)
        
    utk_folder = os.path.join(DEST_DIR, 'UTKFace')
    if os.path.exists(utk_folder):
        for f in os.listdir(utk_folder):
            shutil.move(os.path.join(utk_folder, f), DEST_DIR)
        os.rmdir(utk_folder)
    if os.path.exists(TAR_PATH):
        os.remove(TAR_PATH)
    
    open(os.path.join(DEST_DIR, "modified"), 'w').close()
    print("Pronto!")

In [ ]:
# 3. Preparar Dados
all_files = [os.path.join(DEST_DIR, f) for f in os.listdir(DEST_DIR) if f.endswith('.jpg')]

file_paths = []
ages = []
genders = []

for fpath in all_files:
    fname = os.path.basename(fpath)
    try:
        parts = fname.split('_')
        a = int(parts[0])
        g = int(parts[1])
        if 0 <= a <= 116 and (g==0 or g==1):
            file_paths.append(fpath)
            ages.append(a)
            genders.append(g)
    except:
        continue

train_paths, val_paths, train_ages, val_ages, train_genders, val_genders = train_test_split(
    file_paths, ages, genders, test_size=0.2, random_state=42
)

IMG_SIZE = 128
BATCH_SIZE = 32

def parse_image(filename, age, gender):
    image_string = tf.io.read_file(filename)
    image = tf.io.decode_jpeg(image_string, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, {'gender_output': gender, 'age_output': age}

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_ages, train_genders))
train_ds = train_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_ages, val_genders))
val_ds = val_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 5. Modelo Tuning
def build_model(input_shape):
    inputs = Input(shape=input_shape)
    
    x = Conv2D(32, (3, 3), activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(128, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(256, (3, 3), activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    
    # Gender Branch
    gender_output = Dense(1, activation='sigmoid', name='gender_output')(Dense(128, activation='relu')(x))
    
    # Age Branch 
    x_age = Dense(256, activation='relu')(x) 
    x_age = Dropout(0.2)(x_age)
    age_output = Dense(1, activation='relu', name='age_output')(x_age)
    
    return Model(inputs=inputs, outputs=[gender_output, age_output])

model = build_model((128, 128, 3))

# OTIMIZADOR
opt = tf.keras.optimizers.Adam(learning_rate=0.0001) 

model.compile(optimizer=opt,
              loss={'gender_output': 'binary_crossentropy', 'age_output': 'mse'},
              loss_weights={'gender_output': 1.0, 'age_output': 0.1}, 
              metrics={'gender_output': 'accuracy', 'age_output': 'mae'})
model.summary()

In [ ]:
# 6. Treinar (35 Épocas com Patience=5)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'gender_age_model_v3.h5', 
    monitor='val_gender_output_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# EARLY STOPPING PACIENTE = 5 (MODO FIXED)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_gender_output_accuracy',
    patience=5,
    mode='max', # <<<<<<< CORREÇÃO AQUI (Ensina Keras que Accuracy alta é bom)
    restore_best_weights=True,
    verbose=1
)

history = model.fit(train_ds, 
                    validation_data=val_ds,
                    epochs=35, 
                    callbacks=[checkpoint, early_stop])

In [ ]:
# 7. Baixar
model.save('gender_age_model_v3.h5')

try:
    from google.colab import files
    files.download('gender_age_model_v3.h5')
except:
    print("Baixe manualmenet 'gender_age_model_v3.h5'")